### 1. Persiapan Lingkungan (Setup Environment)

In [ ]:
# Install semua library dari file requirements
!pip install -r requirements.txt --quiet
print("✅ Instalasi berhasil!")

In [ ]:
import duckdb

# Cek versi untuk memastikan instalasi berhasil
print("--- Check Versions ---")
print(f"DuckDB version: {duckdb.__version__}")
print("----------------------")

### 2. Inisialisasi Proyek dbt

In [ ]:
# Membuat folder-folder utama dbt untuk memisahkan lapisan transformasi (Layering)
import os

# Daftar folder standar dbt
folders = [
    'models/staging', 
    'models/dimensions', 
    'models/marts', 
    'seeds', 
    'tests', 
    'macros', 
    'analyses', 
    'snapshots'
]

for folder in folders:
    # exist_ok=True itu fungsinya sama kayak -p (nggak error kalau folder udah ada)
    os.makedirs(folder, exist_ok=True)
    print(f"Directory created: {folder}")
print("✅ Struktur direktori dbt berhasil dibuat.")

In [ ]:
# Menjalankan skrip ingesti data untuk memuat dataset CSV ke dalam DuckDB
!python "load_source.py"

print("✅ Data ingesti selesai. Database 'classicmodels.duckdb' siap digunakan.")

In [ ]:
%%writefile profiles.yml
# Konfigurasi profil koneksi dbt untuk adapter DuckDB
classicmodels_dw:
  outputs:
    dev:
      type: duckdb
      path: 'classicmodels.duckdb'  # Menghubungkan dbt ke file database lokal
      threads: 1                    # Jumlah thread eksekusi paralel
  target: dev

In [ ]:
%%writefile dbt_project.yml
# Konfigurasi utama proyek dbt Classic Models
name: 'classicmodels_dw'
version: '1.0.0'
config-version: 2

# Menghubungkan proyek dengan profil koneksi yang telah dibuat sebelumnya
profile: 'classicmodels_dw'

# Mendefinisikan jalur folder untuk komponen dbt
model-paths: ["models"]
seed-paths: ["seeds"]
test-paths: ["tests"]
analysis-paths: ["analyses"]
macro-paths: ["macros"]
snapshot-paths: ["snapshots"]

# Konfigurasi Materialisasi per Layer
models:
  classicmodels_dw:
    staging:
      +materialized: view    # Abstraksi awal dari source data
    dimensions:
      +materialized: table   # Master data yang dioptimalkan
    marts:
      +materialized: table   # Tabel fakta final untuk kebutuhan analitik

In [ ]:
%%writefile models/sources.yml
version: 2

sources:
  - name: raw_data
    schema: main
    description: "Kumpulan data mentah (raw) dari sistem sumber Classic Models"
    tables:
      - name: customers
        description: "Data profil pelanggan"
      - name: orders
        description: "Informasi transaksi pemesanan"
      - name: orderdetails
        description: "Detail item per pesanan"
      - name: products
        description: "Katalog produk perusahaan"
      - name: employees
        description: "Data internal karyawan"
      - name: offices
        description: "Informasi lokasi kantor cabang"
      - name: payments
        description: "Riwayat transaksi pembayaran pelanggan"
      - name: productlines
        description: "Kategori dan deskripsi lini produk"

In [ ]:
%%writefile models/marts/schema.yml
version: 2

models:
  - name: fct_payments
    description: "Tabel fakta yang mencatat setiap transaksi pembayaran dari pelanggan."
    columns:
      - name: check_number
        description: "ID unik transaksi pembayaran"
        tests:
          - unique
          - not_null

      - name: customer_key
        description: "Kunci asing yang menghubungkan ke tabel dim_customer"
        tests:
          - not_null
          - relationships:
              to: ref('dim_customer')
              field: customer_key

      - name: payment_date_id
        description: "Kunci asing yang menghubungkan ke tabel dim_date"
        tests:
          - not_null
          - relationships:
              to: ref('dim_date')
              field: date_id

      - name: amount
        description: "Nilai nominal pembayaran"
        tests:
          - not_null

In [ ]:
# Menjalankan uji validasi sistem dan koneksi database
!dbt debug --profiles-dir .

### 3. Pengembangan Data Marts dengan dbt

##### Langkah 1: Verifikasi Data Sumber (Source Data Verification)

In [ ]:
import duckdb
import pandas as pd

# Path ke database DuckDB
db_path = 'classicmodels.duckdb'

# Step 1: Memverifikasi daftar tabel yang tersedia di database
print("--- Daftar Tabel dalam Database ---")
with duckdb.connect(db_path) as con:
    df_tables = con.execute("""
        SELECT table_name, table_type
        FROM information_schema.tables
        WHERE table_schema = 'main'
        ORDER BY table_name;
    """).df()
    display(df_tables)

# Step 2: Melakukan preview relasi antar tabel (Order Chain)
print("\n--- Preview Relasi Orders & Customers ---")
query_preview = """
SELECT o.orderNumber,
       o.orderDate,
       o.status,
       c.customerName,
       c.country
FROM orders o
JOIN customers c USING (customerNumber)
LIMIT 10;
"""

with duckdb.connect(db_path) as con:
    df_preview = con.execute(query_preview).df()
    display(df_preview)

##### Langkah 2: Membangun Staging Layer

In [ ]:
%%writefile models/staging/stg_orders.sql
select
    orderNumber    as order_id,
    orderDate      as order_date,
    requiredDate   as required_date,
    shippedDate    as shipped_date,
    status         as order_status,
    comments       as comments,
    customerNumber as customer_id
from {{ source('raw_data', 'orders') }}

In [ ]:
%%writefile models/staging/stg_orderdetails.sql
select
    orderNumber      as order_id,
    productCode      as product_code,
    quantityOrdered  as quantity_ordered,
    priceEach        as price_each,
    orderLineNumber  as order_line_number
from {{ source('raw_data', 'orderdetails') }}

In [ ]:
%%writefile models/staging/stg_customers.sql
SELECT
    customerNumber AS customer_id,
    customerName AS customer_name,
    contactLastName AS contact_last_name,
    contactFirstName AS contact_first_name,
    phone,
    salesRepEmployeeNumber AS sales_rep_employee_id,
    city,
    country
FROM {{ source('raw_data', 'customers') }}

In [ ]:
%%writefile models/staging/stg_products.sql
SELECT
    productCode AS product_code,
    productName AS product_name,
    productLine AS product_line,
    productScale AS product_scale,
    productVendor AS product_vendor,
    quantityInStock AS quantity_in_stock,
    buyPrice AS buy_price,
    MSRP AS msrp
FROM {{ source('raw_data', 'products') }}

In [ ]:
%%writefile models/staging/stg_employees.sql
SELECT
    employeeNumber AS employee_id,
    lastName AS last_name,
    firstName AS first_name,
    extension,
    email,
    officeCode AS office_code,
    reportsTo AS reports_to_employee_id,
    jobTitle AS job_title
FROM {{ source('raw_data', 'employees') }}

In [ ]:
%%writefile models/staging/stg_offices.sql
SELECT
    officeCode AS office_code,
    city,
    phone,
    addressLine1 AS address_line_1,
    country,
    territory
FROM {{ source('raw_data', 'offices') }}

In [ ]:
%%writefile models/staging/stg_payments.sql
SELECT
    customerNumber AS customer_id,
    checkNumber AS check_number,
    paymentDate AS payment_date,
    amount
FROM {{ source('raw_data', 'payments') }}

In [ ]:
%%writefile models/staging/stg_productlines.sql
SELECT
    productLine AS product_line,
    textDescription AS text_description
FROM {{ source('raw_data', 'productlines') }}

In [ ]:
# run dbt staging
!dbt run --profiles-dir . --select staging

In [ ]:
# checkpoint
query = """SELECT order_id, order_date, order_status, customer_id
FROM stg_orders
LIMIT  5;"""

with duckdb.connect(db_path) as con:
    # Cek apakah view stg_orders sudah muncul
    df = con.execute(query).df()
    display(df)

query = """SELECT order_id, product_code, quantity_ordered, price_each
FROM stg_orderdetails
LIMIT 5;"""

with duckdb.connect(db_path) as con:
    # Cek apakah view stg_orders sudah muncul
    df = con.execute(query).df()
    display(df)


##### Langkah 3: Membangun Model Dimensi

In [ ]:
%%writefile models/dimensions/dim_date.sql
with date_spine as (
    -- Date range covers all ClassicModels order dates.
    -- Extended to full calendar years for clean reporting.
    select generate_series::date as date_day
    from generate_series(
        date '2003-01-01',
        date '2006-12-31',
        interval '1 day'
    )
)
select
    date_day                                as date_id,
    date_day                                as full_date,
    date_trunc('month', date_day)::date     as date_month,
    extract('year'    from date_day)::int   as year,
    extract('quarter' from date_day)::int   as quarter,
    extract('month'   from date_day)::int   as month,
    strftime('%B', date_day)                as month_name,
    extract('week'    from date_day)::int   as week_of_year,
    extract('day'     from date_day)::int   as day_of_month,
    extract('dow'     from date_day)::int   as day_of_week,
    strftime('%A', date_day)                as day_name,
    (extract('dow' from date_day) in (0, 6)) as is_weekend
from date_spine

In [ ]:
%%writefile models/dimensions/dim_product.sql
select
    row_number() over (order by p.product_code)   as product_key,
    p.product_code,
    p.product_name,
    p.product_line,
    pl.text_description                           as product_line_description,
    p.product_scale,
    p.product_vendor,
    p.buy_price,
    p.msrp
from {{ ref('stg_products') }} p
left join {{ ref('stg_productlines') }} pl
    on p.product_line = pl.product_line

In [ ]:
%%writefile models/dimensions/dim_employee.sql
select
    row_number() over (order by e.employee_id) as employee_key,
    e.employee_id,
    e.first_name,
    e.last_name,
    e.first_name || ' ' || e.last_name as full_name,
    e.job_title,
    e.email,
    e.office_code,
    e.reports_to_employee_id,
    mgr.first_name || ' ' || mgr.last_name as manager_full_name
from {{ ref('stg_employees') }} e
left join {{ ref('stg_employees') }} mgr
    on e.reports_to_employee_id = mgr.employee_id

In [ ]:
%%writefile models/dimensions/dim_customer.sql
SELECT
    row_number() over (order by customer_id) as customer_key,
    customer_id,
    customer_name,
    contact_first_name || ' ' || contact_last_name AS full_contact_name,
    phone,
    sales_rep_employee_id,
    city,
    country
FROM {{ ref('stg_customers') }}



In [ ]:
%%writefile models/dimensions/dim_office.sql
SELECT
    row_number() over (order by office_code) as office_key,
    office_code,
    city,
    phone,
    address_line_1,
    country,
    territory
FROM {{ ref('stg_offices') }}

In [ ]:
!dbt run --profiles-dir . --select dimensions

In [ ]:
# 1. Audit Dimensi Waktu
print("--- Verifikasi Rentang Data dim_date ---")
query_date = """
SELECT
    MIN(date_id) AS start_date,
    MAX(date_id) AS end_date,
    COUNT(*) AS total_days
FROM dim_date;
"""

with duckdb.connect(db_path) as con:
    df_date = con.execute(query_date).df()
    display(df_date)
    # Catatan: Pastikan jumlah hari sesuai dengan kalender (termasuk kabisat)

# 2. Audit Hirarki Karyawan
print("\n--- Verifikasi Hirarki Manajer dim_employee ---")
query_emp = """
SELECT
    full_name,
    job_title,
    manager_full_name
FROM dim_employee
ORDER BY employee_id;
"""

with duckdb.connect(db_path) as con:
    df_emp = con.execute(query_emp).df()
    display(df_emp)

##### Langkah 4: Build the Fact Tables

In [ ]:
%%writefile models/marts/fct_order_lines.sql
select
    od.order_id,
    od.order_line_number,
    o.order_date                           as order_date_id,
    dc.customer_key,
    dp.product_key,
    de.employee_key                        as sales_rep_key,
    dof.office_key,
    od.quantity_ordered,
    od.price_each,
    round(od.quantity_ordered * od.price_each, 2) as revenue,
    dp.buy_price,
    round(od.quantity_ordered * dp.buy_price, 2) as cost,
    round(
        (od.quantity_ordered * od.price_each)
      - (od.quantity_ordered * dp.buy_price), 2
    ) as gross_profit
from {{ ref('stg_orderdetails') }} od
join {{ ref('stg_orders') }} o
    on od.order_id = o.order_id
join {{ ref('dim_customer') }} dc
    on o.customer_id = dc.customer_id
join {{ ref('dim_product') }} dp
    on od.product_code = dp.product_code
left join {{ ref('dim_employee') }} de
    on dc.sales_rep_employee_id = de.employee_id
left join {{ ref('dim_office') }} dof
    on de.office_code = dof.office_code

In [ ]:
%%writefile models/marts/fct_payments.sql
-- depends_on: {{ ref('dim_date') }}
select
    p.check_number,
    p.payment_date as payment_date_id,
    dc.customer_key,
    p.amount
from {{ ref('stg_payments') }} p
join {{ ref('dim_customer') }} dc
    on p.customer_id = dc.customer_id

In [ ]:
%%writefile models/marts/fct_sales_performance.sql
select
    sales_rep_key,
    office_key,
    dd.month_date                                as month_date,
    dd.year                                      as year,
    dd.month                                     as month,
    count(distinct order_id)                     as order_count,
    count(distinct customer_key)                 as unique_customers,
    sum(quantity_ordered)                        as total_units_sold,
    round(sum(revenue), 2)                       as total_revenue,
    round(sum(gross_profit), 2)                  as total_gross_profit,
    round(avg(revenue), 2)                       as avg_order_line_revenue
from {{ ref('fct_order_lines') }} fol 
left join {{ref('dim_date')}} dd 
    on fol.order_date_id=dd.date_id
-- Orders with no assigned sales rep are excluded (rep-level analysis only).
where sales_rep_key is not null
group by 1, 2, 3, 4, 5

In [ ]:
# Mengeksekusi seluruh model marts
!dbt run --profiles-dir . --select marts

##### Langkah 5: Validasi Kualitas Data (Data Testing)

In [ ]:
# Menjalankan pengujian kualitas data khusus untuk tabel fct_payments
!dbt test --select fct_payments --profiles-dir .

print("\n✅ Validasi integritas tabel fct_payments selesai.")

##### Langkah 6: Eksplorasi Data Marts & Analisis Bisnis

###### Skenario 1: Identifikasi Produk "High-Volume, Low-Margin"

In [ ]:
query = """
SELECT
    p.product_name,
    SUM(f.quantity_ordered) AS total_units_sold,
    ROUND(SUM(f.revenue), 2) AS total_revenue,
    ROUND(SUM(f.gross_profit), 2) AS total_profit,
    ROUND(SUM(f.gross_profit) / NULLIF(SUM(f.revenue), 0) * 100, 2) AS profit_margin_pct
FROM 'fct_order_lines' f
JOIN 'dim_product' p ON f.product_key = p.product_key
GROUP BY 1
ORDER BY total_units_sold DESC
LIMIT 10;
"""
with duckdb.connect(db_path) as con:
    df = con.execute(query).df()
    display(df)

###### Skenario 2: Analisis Likuiditas (Sales vs Payment)

In [ ]:
# Query ini membandingkan akumulasi pesanan pelanggan dengan realisasi pembayaran yang sudah diterima.

# Identifikasi Produk "High-Volume, Low-Margin"
query = """
WITH customer_sales AS (
    SELECT
        customer_key,
        SUM(revenue) AS total_ordered
    FROM fct_order_lines
    GROUP BY 1
),
customer_payments AS (
    SELECT
        customer_key,
        SUM(amount) AS total_paid
    FROM fct_payments
    GROUP BY 1
)
SELECT
    c.customer_name,
    s.total_ordered,
    COALESCE(p.total_paid, 0) AS total_paid,
    ROUND((s.total_ordered - COALESCE(p.total_paid, 0)),2) AS outstanding_balance
FROM customer_sales s
LEFT JOIN customer_payments p ON s.customer_key = p.customer_key
JOIN dim_customer c ON s.customer_key = c.customer_key
where outstanding_balance > 0
ORDER BY outstanding_balance DESC;
"""
with duckdb.connect(db_path) as con:
    df = con.execute(query).df()
    display(df)


###### Skenario 3: Evaluasi Efisiensi Sales Representative

In [ ]:
# Query ini mencari sales yang paling efisien dalam menghasilkan profit per transaksi.
query = """
SELECT
    e.first_name || ' ' || e.last_name AS sales_rep_name,
    SUM(order_count) AS total_orders,
    SUM(total_revenue) AS total_revenue,
    SUM(total_gross_profit) AS total_profit,
    ROUND(SUM(total_gross_profit) / SUM(order_count), 2) AS avg_profit_per_order
FROM fct_sales_performance f
JOIN dim_employee e ON f.sales_rep_key = e.employee_key
GROUP BY 1
ORDER BY avg_profit_per_order DESC;
"""
with duckdb.connect(db_path) as con:
    df = con.execute(query).df()
    display(df)


###### Skenario 4: Pemetaan Tren Penjualan Musiman

In [ ]:
# Query ini menggunakan tabel dimensi waktu untuk melihat performa per kuartal secara historis.
query = """
SELECT
    d.year,
    d.quarter,
    ROUND(SUM(f.revenue), 2) AS periodic_revenue,
    ROUND(SUM(f.gross_profit), 2) AS periodic_profit
FROM fct_order_lines f
JOIN dim_date d ON f.order_date_id = d.date_id
GROUP BY 1, 2
ORDER BY 1, 2;
"""
with duckdb.connect(db_path) as con:
    df = con.execute(query).df()
    display(df)


###### Skenario 5: Optimalisasi Operasional Kantor Cabang

In [ ]:
# Query ini memantau produktivitas kantor cabang berdasarkan tren bulanan.
query = """
SELECT
    o.city AS office_location,
    f.month_date,
    f.unique_customers,
    f.total_revenue,
    f.total_gross_profit
FROM fct_sales_performance f
JOIN dim_office o ON f.office_key = o.office_key
ORDER BY 1, 2;
"""
with duckdb.connect(db_path) as con:
    df = con.execute(query).df()
    display(df)


##### Langkah 7: Visualisasi Silsilah Data (dbt DAG)

In [ ]:
# Menghasilkan metadata dokumentasi proyek
!dbt docs generate --profiles-dir .

In [ ]:
"""
Jika menjalankan secara Lokal (Laptop/PC):
Gunakan perintah standar dbt di terminal untuk membuka server lokal di
http://localhost:8080.
"""

!dbt docs serve --profiles-dir .

In [ ]:
# """
# Jika menjalankan di Google Colab:
# Karena Colab berjalan di server cloud Google,
# kita perlu menggunakan proxy internal agar browser Anda bisa mengakses folder target yang ada di server tersebut.
# """
# from google.colab import output
# import os

# # Pastikan folder target ada
# if os.path.exists("target"):
#     # Jalankan server internal
#     port = 8000
#     get_ipython().system_raw(f'python3 -m http.server {port} --directory target &')

#     # Berikan link proxy internal Google
#     print("Klik link di bawah ini untuk melihat docs:")
#     print(output.eval_js(f'google.colab.kernel.proxyPort({port})'))
# else:
#     print("Folder 'target' tidak ditemukan. Jalankan '!dbt docs generate' dulu!")